# Example 07 — FE Euler-Bernoulli Beam with Tanh Dry Friction

FRF near first bending resonance and spatial mode shape of a clamped-free Euler-Bernoulli beam with tanh dry friction at the midpoint node, using HB arc-length continuation. (Krack & Gross 2019, §5)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.linalg import eigh

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.systems.fe_beam import FE_EulerBernoulliBeam
from nlvib.nonlinearities.elements import tanh_dry_friction
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

In [ ]:
# --- System setup ---
N_ELEMENTS = 10; L_BEAM = 1.0; E_MOD = 2.1e11; I_AREA = 1e-8; RHO = 7800.0; A_SECT = 1e-4
FRICTION_F0 = 5.0; FRICTION_C = 100.0; FRICTION_NODE = 5
FORCE_AMP = 10.0; FORCE_NODE = N_ELEMENTS
N_HARMONICS = 3; OMEGA_MIN = 150.0; OMEGA_MAX = 250.0

beam = FE_EulerBernoulliBeam(n_elements=N_ELEMENTS, L=L_BEAM, E=E_MOD,
                              I_area=I_AREA, rho=RHO, A=A_SECT, bc='clamped-free')
midpoint_dof = beam.find_dof(FRICTION_NODE, 'w')
nl_element = tanh_dry_friction(f0=FRICTION_F0, c=FRICTION_C, dof_index=midpoint_dof)
beam.add_nonlinear_attachment(FRICTION_NODE, 'w', nl_element)
beam.add_forcing(FORCE_NODE, 'w', FORCE_AMP)
tip_dof = beam.find_dof(FORCE_NODE, 'w')
excitation = {'dof': tip_dof, 'amplitude': FORCE_AMP, 'harmonic': 1}

K_dense = beam.K.toarray(); M_dense = beam.M.toarray()
eigvals, eigvecs = eigh(K_dense, M_dense)
omega1_linear = float(np.sqrt(np.abs(eigvals[0])))
print(f'Linear first natural frequency: {omega1_linear:.2f} rad/s')

if not (OMEGA_MIN <= omega1_linear <= OMEGA_MAX):
    half_width = max(50.0, 0.3 * omega1_linear)
    omega_start = max(1.0, omega1_linear - half_width)
    omega_end = omega1_linear + half_width
    print(f'Adjusting frequency range to [{omega_start:.1f}, {omega_end:.1f}] rad/s')
else:
    omega_start = OMEGA_MIN; omega_end = OMEGA_MAX

In [ ]:
# --- Initial solution + continuation ---
n_dof = beam.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)
Q0 = np.zeros(n_total)
for _ in range(40):
    R, J = hb_residual(Q0, omega_start, beam, N_HARMONICS, excitation)
    if np.linalg.norm(R) < 1e-8: break
    try: dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    step_norm = np.linalg.norm(dQ)
    if step_norm > 10.0: dQ *= 10.0 / step_norm
    Q0 += dQ
print(f'Initial residual at omega={omega_start:.1f}: {np.linalg.norm(R):.3e}')

def residual_fn(Q, omega):
    return hb_residual(Q, omega, beam, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=0.5, ds_min=1e-4, ds_max=5.0, max_steps=800,
    newton_tol=1e-6, lambda_min=omega_start, lambda_max=omega_end,
)
result = ContinuationSolver().run(residual_fn, Q0, omega_start, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract FRF at tip and save plots ---
solutions = result.solutions
omegas = solutions[:, -1]
stability = result.stability
cos1_tip = solutions[:, n_dof*1 + tip_dof]
sin1_tip = solutions[:, n_dof*2 + tip_dof]
amp_tip = np.sqrt(cos1_tip**2 + sin1_tip**2)

if len(amp_tip) > 0:
    peak_idx = int(np.argmax(amp_tip))
    peak_amp_tip = float(amp_tip[peak_idx])
    resonance_freq = float(omegas[peak_idx])
else:
    peak_idx = 0; peak_amp_tip = resonance_freq = float('nan')

output_dir = Path('..') / 'examples' / '07_beam_tanh_friction' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig_frf, ax_frf = plt.subplots(figsize=(8, 5))
for i in range(len(omegas) - 1):
    is_stable = not bool(stability[i])
    ax_frf.plot(omegas[i:i+2], amp_tip[i:i+2],
                color='tab:blue' if is_stable else 'tab:red',
                linestyle='-' if is_stable else '--', linewidth=1.5)
handles = [
    Line2D([0], [0], color='tab:blue', linestyle='-', label='stable'),
    Line2D([0], [0], color='tab:red', linestyle='--', label='unstable'),
]
ax_frf.legend(handles=handles)
ax_frf.set_xlabel(r'Excitation frequency $\Omega$ (rad/s)')
ax_frf.set_ylabel(r'Tip amplitude $|w_{\rm tip}|$ (harmonic 1)')
ax_frf.set_title('Example 07 — Beam Tanh Friction FRF (tip)')
ax_frf.axvline(resonance_freq, color='gray', linestyle=':', linewidth=0.8)
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf.png', dpi=150)
print('Saved frf.png')

# Mode shape at resonance peak
Q_peak = solutions[peak_idx, :n_total]
node_positions = [0.0]; mode_shape_amp = [0.0]
for node_i in range(1, N_ELEMENTS + 1):
    try:
        r_dof = beam.find_dof(node_i, 'w')
        c1 = Q_peak[n_dof*1 + r_dof]; s1 = Q_peak[n_dof*2 + r_dof]
        node_positions.append(node_i * L_BEAM / N_ELEMENTS)
        mode_shape_amp.append(float(np.sqrt(c1**2 + s1**2)))
    except ValueError: pass

node_positions_arr = np.array(node_positions); mode_shape_arr = np.array(mode_shape_amp)
fig_mode, ax_mode = plt.subplots(figsize=(8, 4))
ax_mode.plot(node_positions_arr, np.zeros_like(node_positions_arr), 'k--', linewidth=0.6, label='undeformed')
ax_mode.plot(node_positions_arr, mode_shape_arr, 'o-', color='tab:green', linewidth=1.5,
             label=f'mode shape at resonance ({resonance_freq:.1f} rad/s)')
ax_mode.fill_between(node_positions_arr, mode_shape_arr, alpha=0.2, color='tab:green')
ax_mode.set_xlabel('Position along beam (m)'); ax_mode.set_ylabel(r'Transverse amplitude (m)')
ax_mode.set_title('Example 07 — Mode Shape at Resonance Peak')
ax_mode.legend(); ax_mode.grid(True, alpha=0.3); fig_mode.tight_layout()
fig_mode.savefig(output_dir / 'mode_shape.png', dpi=150)
print('Saved mode_shape.png')

In [ ]:
# --- Display FRF inline ---
fig_frf

In [ ]:
# --- Display mode shape inline ---
fig_mode

In [ ]:
# --- Summary ---
print('=' * 55)
print('SUMMARY — Example 07: Beam Tanh Friction')
print('=' * 55)
print(f'  Linear 1st natural frequency : {omega1_linear:.2f} rad/s')
print(f'  Resonance frequency (HB)     : {resonance_freq:.2f} rad/s')
print(f'  Tip amplitude at resonance   : {peak_amp_tip:.6e} m')
print('=' * 55)